## **04 MongoDB Python Advance**

### 0. 기본 pymongo 템플릿 코드
> sample_mflix 데이터셋을 기반으로, 지금까지 익힌 mongodb 문법을 pymongo 에서 어떻게 적용해서 사용할 수 있는지를 알아보기로 함

In [69]:
from pymongo import MongoClient
client = MongoClient("mongodb://localhost:27017/") 

# db = client['sample_mflix']
db = client.sample_mflix
movies = db.movies
movies

Collection(Database(MongoClient(host=['localhost:27017'], document_class=dict, tz_aware=False, connect=True), 'sample_mflix'), 'movies')

In [15]:
movies.find()

### 다양한 find() 문법 적용

**1. 프로젝션(projection) - 결과 문서에 표시할 필드 지정:**

In [37]:
results = movies.find({"year": 1923}, {"_id" : 0, "title" : 1, "year": 1})
results

In [34]:
for movie in results:
    print(movie)

{'title': 'The Hunchback of Notre Dame', 'year': 1923}
{'title': 'Our Hospitality', 'year': 1923}
{'title': 'Safety Last!', 'year': 1923}
{'title': 'Three Ages', 'year': 1923}
{'title': 'A Woman of Paris: A Drama of Fate', 'year': 1923}
{'title': 'The Chechahcos', 'year': 1923}


In [35]:
movie, type(movie) # dictionary 형태로 출력

({'title': 'The Chechahcos', 'year': 1923}, dict)

**2. 비교 쿼리 연산자 - MongoDB 비교 쿼리 연산자 사용:**

In [40]:
# 1910년 이전에 출시된 영화의 제목, 개봉 연도, 감독을 조회
results = movies.find({"years":{"$lt":1910}},{"_id":0, "title":1, "year":1, "director":1})
for movie in results:
    print(movie)

**3. 논리 쿼리 연산자 - MongoDB 논리 쿼리 연산자 사용:**

In [42]:
# 1900년 이전 또는 2015년 이후에 출시된 영화 찾기 (title, year, plot 필드만 조회)
results = movies.find({"$or":[{"years":{"$lt":1900}}, {"years":{"$gt":2015}}]},{"_id":0, "title":1, "year":1, "plot":1})
list(results)

[]

**4. 배열 쿼리 연산자 - MongoDB 배열 쿼리 연산자 사용:**

In [27]:
# 'Action'과 'Sci-Fi' (['Action', 'Sci-Fi]) 장르의 영화 찾기 (title, 연도, 장르만)
results = movies.find({"genres": {"$all": ["Action", "Sci-Fi"]}}, 
                                 {"_id" : 0, "title": 1, "year": 1, "genres": 1})
list(results)

[{'genres': ['Action', 'Adventure', 'Sci-Fi'],
  'title': 'Flash Gordon',
  'year': 1936},
 {'genres': ['Action', 'Sci-Fi', 'Thriller'],
  'title': 'The War of the Worlds',
  'year': 1953},
 {'genres': ['Action', 'Sci-Fi'], 'title': 'The 10th Victim', 'year': 1965},
 {'genres': ['Action', 'Drama', 'Sci-Fi'],
  'title': 'Das Millionenspiel',
  'year': 1970},
 {'genres': ['Action', 'Sci-Fi'],
  'title': 'Battle for the Planet of the Apes',
  'year': 1973},
 {'genres': ['Action', 'Sci-Fi', 'Thriller'],
  'title': 'Westworld',
  'year': 1973},
 {'genres': ['Action', 'Sci-Fi', 'Sport'],
  'title': 'Rollerball',
  'year': 1975},
 {'genres': ['Action', 'Adventure', 'Sci-Fi'],
  'title': 'Buck Rogers in the 25th Century',
  'year': 1979},
 {'year': 1978, 'genres': ['Action', 'Drama', 'Sci-Fi'], 'title': 'Superman'},
 {'genres': ['Action', 'Adventure', 'Sci-Fi'],
  'title': 'Message from Space',
  'year': 1978},
 {'genres': ['Action', 'Adventure', 'Sci-Fi'],
  'title': 'Mad Max',
  'year': 1979

**5. 정렬하기(sort), 앞쪽 일부 건너뛰기(skip), 갯수 제한하기(limit):**
- find() 에 붙여서, 별도 메서드로 사용

In [30]:
# IMDB rating으로 내림차순 정렬, 최고점 3개 skip 5건 가져오기
result = movies.find({"imdb.rating":{"$ne":""}}).sort({"imdb.rating":-1}).skip(3).limit(5)
list(result)

[{'_id': ObjectId('573a1398f29313caabcebc0b'),
  'plot': 'A comprehensive survey of the American Civil War.',
  'genres': ['Documentary', 'History', 'War'],
  'runtime': 680,
  'cast': ['Sam Waterston', 'Julie Harris', 'Jason Robards', 'Morgan Freeman'],
  'num_mflix_comments': 2,
  'poster': 'https://m.media-amazon.com/images/M/MV5BZDc1NzI2MGEtZDA2Yy00ZWExLTgwYmItNjU3N2QyYmM0MzYwXkEyXkFqcGdeQXVyNTA4NzY1MzY@._V1_SY1000_SX677_AL_.jpg',
  'title': 'The Civil War',
  'fullplot': "This highly acclaimed mini series traces the course of the U.S. Civil War from the abolitionist movement through all the major battles to the death of President Lincoln and the beginnings of Reconstruction. The story is mostly told in the words of the participants themselves, through their diaries, letters, and Visuals are usually still photographs and illustrations of the time, and the soundtrack is likewise made up of war-era tunes played on period instruments. Several modern-day historians offer periodic comme

**6. 정규표현식과 pymongo**

-  파이썬의 정규표현식 라이브러리인 `re` 모듈의 `compile` 함수를 사용하여 정규 표현식 객체를 생성하고,
- 이를 pymongo 에 적용할 수 있습니다.

- 예: re.I (IGNORECASE): 이 옵션은 대소문자를 구분하지 않는다는 것을 나타냅니다. 따라서 'Star', 'STAR', 'star', 'sTaR' 등을 모두 찾을 수 있습니다.

In [ ]:
import re # regular expression 모듈
regex = re.compile("Star", re.I) # I : ignore - "Star"라는 단어가 포함된 영화 제목을 대소문자 구분 없이 검색
# for movie in movies.find({"title": regex}, {"_id":0, "title": 1}):
#     print(movie)

movies_list =list(movies.find({"title":regex}))

[{'_id': ObjectId('573a1392f29313caabcdb497'),
  'plot': 'A young woman comes to Hollywood with dreams of stardom, but achieves them only with the help of an alcoholic leading man whose best days are behind him.',
  'genres': ['Drama'],
  'runtime': 111,
  'rated': 'NOT RATED',
  'cast': ['Janet Gaynor', 'Fredric March', 'Adolphe Menjou', 'May Robson'],
  'poster': 'https://m.media-amazon.com/images/M/MV5BMmE5ODI0NzMtYjc5Yy00MzMzLTk5OTQtN2Q3MzgwOTllMTY3XkEyXkFqcGdeQXVyNjc0MzMzNjA@._V1_SY1000_SX677_AL_.jpg',
  'title': 'A Star Is Born',
  'fullplot': 'Esther Blodgett is just another starry-eyed farm kid trying to break into the movies. Waitressing at a Hollywood party, she catches the eye of alcoholic star Norman Maine, is given a test, and is caught up in the Hollywood glamor machine (ruthlessly satirized). She and her idol Norman marry; but his career abruptly dwindles to nothing',
  'languages': ['English'],
  'released': datetime.datetime(1937, 4, 27, 0, 0),
  'directors': ['William

In [49]:
# !pip install pandas

In [51]:
import pandas as pd
df = pd.DataFrame(movies_list)
df

,_id,plot,genres,runtime,rated,cast,poster,title,fullplot,languages,...,writers,awards,lastupdated,year,imdb,countries,type,tomatoes,num_mflix_comments,metacritic
0,573a1392f29313caabcdb497,A young woman comes to Hollywood with dreams o...,[Drama],111.0,NOT RATED,"[Janet Gaynor, Fredric March, Adolphe Menjou, ...",https://m.media-amazon.com/images/M/MV5BMmE5OD...,A Star Is Born,Esther Blodgett is just another starry-eyed fa...,[English],...,"[Dorothy Parker (screen play), Alan Campbell (...","{'wins': 3, 'nominations': 7, 'text': 'Won 1 O...",2015-09-01 00:55:54.333000000,1937,"{'rating': 7.7, 'votes': 5005, 'id': 29606}",[USA],movie,{'website': 'http://www.vcientertainment.com/F...,NaN,NaN
1,573a1392f29313caabcdbdd3,Davey Fenwick leaves his mining village on a u...,[Drama],110.0,NaN,"[Michael Redgrave, Margaret Lockwood, Emlyn Wi...",https://m.media-amazon.com/images/M/MV5BNzI0OD...,The Stars Look Down,Davey Fenwick leaves his mining village on a u...,[English],...,"[A.J. Cronin (from the book by), J.B. Williams...","{'wins': 1, 'nominations': 0, 'text': '1 win.'}",2015-09-02 00:42:27.583000000,1940,"{'rating': 7.2, 'votes': 634, 'id': 31976}",[UK],movie,"{'viewer': {'rating': 3.5, 'numReviews': 330, ...",1.0,NaN
2,573a1394f29313caabcdfa75,A film star helps a young singer and actress f...,"[Drama, Musical, Romance]",154.0,APPROVED,"[Judy Garland, James Mason, Jack Carson, Charl...",https://m.media-amazon.com/images/M/MV5BNTg2Mz...,A Star Is Born,"Norman Maine, a movie star whose career is on ...",[English],...,"[Moss Hart (screenplay), Dorothy Parker, Alan ...","{'wins': 12, 'nominations': 3, 'text': 'Nomina...",2015-09-13 00:36:10.993000000,1954,"{'rating': 7.8, 'votes': 10141, 'id': 47522}",[USA],movie,"{'viewer': {'rating': 3.7, 'numReviews': 8985,...",1.0,NaN
3,573a1394f29313caabce0845,A cynical former sheriff turned bounty hunter ...,[Western],93.0,APPROVED,"[Henry Fonda, Anthony Perkins, Betsy Palmer, M...",https://m.media-amazon.com/images/M/MV5BZTdkMG...,The Tin Star,Veteran bounty-hunter Morg Hickman rides into ...,[English],...,"[Joel Kane (story), Dudley Nichols, Barney Sla...","{'wins': 0, 'nominations': 2, 'text': 'Nominat...",2015-08-25 00:03:12.260000000,1957,"{'rating': 7.4, 'votes': 3260, 'id': 51087}",[USA],movie,"{'viewer': {'rating': 3.5, 'numReviews': 1458,...",1.0,NaN
4,573a1394f29313caabce0b7a,"While working as a counselor at a summer camp,...","[Drama, Romance]",128.0,APPROVED,"[Gene Kelly, Natalie Wood, Claire Trevor, Ever...",https://m.media-amazon.com/images/M/MV5BOWU2ZT...,Marjorie Morningstar,"While working as a counselor at a summer camp,...",[English],...,"[Everett Freeman (screenplay), Herman Wouk (no...","{'wins': 5, 'nominations': 2, 'text': 'Nominat...",2015-09-17 04:41:14.380000000,1958,"{'rating': 6.3, 'votes': 686, 'id': 51911}",[USA],movie,"{'viewer': {'rating': 3.2, 'numReviews': 1192,...",NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
137,573a13e2f29313caabdbe91f,Marco returns to Paris after his brother-in-la...,[Drama],100.0,UNRATED,"[Vincent Lindon, Chiara Mastroianni, Julie Bat...",https://m.media-amazon.com/images/M/MV5BMTAxNz...,Bastards,Supertanker captain Marco Silvestri is called ...,"[French, English]",...,"[Jean-Pol Fargeau (screenplay), Claire Denis (...","{'wins': 1, 'nominations': 1, 'text': '1 win &...",2015-08-07 00:24:11.960000000,2013,"{'rating': 6.1, 'votes': 2031, 'id': 2821088}","[France, Germany]",movie,{'website': 'http://www.ifcfilms.com/films/bas...,1.0,69.0
138,573a13e7f29313caabdc70ea,"""ALL-STARS"" follows a girls' 10-year-old fastp...",[Comedy],98.0,NaN,"[Rose Abdoo, Jenica Bergere, Illeana Douglas, ...",https://m.media-amazon.com/images/M/MV5BMjA0Mj...,All Stars,"""ALL-STARS"" follows a girls' 10-year-old fastp...",[English],...,"[Lance Kinsey, Lance Kinsey]","{'wins': 1, 'nominations': 1, 'text': '1 win &...",2015-09-02 00:22:19.287000000,2014,"{'rating': 8.4, 'votes': 73, 'id': 3113448}",[USA],movie,"{'view

In [77]:
from pymongo import MongoClient
client = MongoClient("mongodb://localhost:27017/")  # MongoDB 연결

db = client.sample_mflix  # 데이터베이스 선택
# db = client['sample_mflix']  # 데이터베이스 선택 (위와 동일)
movies = db.movies
movies   # sample_mflix db의 movies컬렉션 객체 (collection object)

Collection(Database(MongoClient(host=['localhost:27017'], document_class=dict, tz_aware=False, connect=True), 'sample_mflix'), 'movies')

- re 모듈 없이, 직접 정규표현식을 pymongo 에 사용할 수도 있음
- `$options`
   - `i`: 대소문자를 구분하지 않습니다. (re 라이브러리에서는 re.I)

In [78]:
regex

re.compile(r'Star', re.IGNORECASE|re.UNICODE)

In [57]:
list(movies.find({"title": regex}).limit(1)) # db.movies.find({title: /Star/i}).litmi(1)과 동일한 결과를 반환

[{'_id': ObjectId('573a1392f29313caabcdb497'),
  'plot': 'A young woman comes to Hollywood with dreams of stardom, but achieves them only with the help of an alcoholic leading man whose best days are behind him.',
  'genres': ['Drama'],
  'runtime': 111,
  'rated': 'NOT RATED',
  'cast': ['Janet Gaynor', 'Fredric March', 'Adolphe Menjou', 'May Robson'],
  'poster': 'https://m.media-amazon.com/images/M/MV5BMmE5ODI0NzMtYjc5Yy00MzMzLTk5OTQtN2Q3MzgwOTllMTY3XkEyXkFqcGdeQXVyNjc0MzMzNjA@._V1_SY1000_SX677_AL_.jpg',
  'title': 'A Star Is Born',
  'fullplot': 'Esther Blodgett is just another starry-eyed farm kid trying to break into the movies. Waitressing at a Hollywood party, she catches the eye of alcoholic star Norman Maine, is given a test, and is caught up in the Hollywood glamor machine (ruthlessly satirized). She and her idol Norman marry; but his career abruptly dwindles to nothing',
  'languages': ['English'],
  'released': datetime.datetime(1937, 4, 27, 0, 0),
  'directors': ['William

In [71]:
for movie in movies.find({"title": {"$regex": "star", "$options": 'i'}}).limit(1): 
    print(movie)

{'_id': ObjectId('573a1392f29313caabcdb497'), 'plot': 'A young woman comes to Hollywood with dreams of stardom, but achieves them only with the help of an alcoholic leading man whose best days are behind him.', 'genres': ['Drama'], 'runtime': 111, 'rated': 'NOT RATED', 'cast': ['Janet Gaynor', 'Fredric March', 'Adolphe Menjou', 'May Robson'], 'poster': 'https://m.media-amazon.com/images/M/MV5BMmE5ODI0NzMtYjc5Yy00MzMzLTk5OTQtN2Q3MzgwOTllMTY3XkEyXkFqcGdeQXVyNjc0MzMzNjA@._V1_SY1000_SX677_AL_.jpg', 'title': 'A Star Is Born', 'fullplot': 'Esther Blodgett is just another starry-eyed farm kid trying to break into the movies. Waitressing at a Hollywood party, she catches the eye of alcoholic star Norman Maine, is given a test, and is caught up in the Hollywood glamor machine (ruthlessly satirized). She and her idol Norman marry; but his career abruptly dwindles to nothing', 'languages': ['English'], 'released': datetime.datetime(1937, 4, 27, 0, 0), 'directors': ['William A. Wellman', 'Jack Con

**7. distinct: 이 메소드는 특정 필드의 모든 고유한 값을 반환합니다.**

In [72]:
print(movies.distinct("genres"))

['Action', 'Adventure', 'Animation', 'Biography', 'Comedy', 'Crime', 'Documentary', 'Drama', 'Family', 'Fantasy', 'Film-Noir', 'History', 'Horror', 'Music', 'Musical', 'Mystery', 'News', 'Romance', 'Sci-Fi', 'Short', 'Sport', 'Talk-Show', 'Thriller', 'War', 'Western']


**8. $in: 이 연산자는 필드 값이 특정 배열 내의 값 중 하나와 일치하는 문서를 선택합니다.**

In [73]:
results = movies.find({"genres": {"$in": ["Action", "Adventure"]}},{"title":1,"genres":1}).limit(3)
results

In [76]:
for movie in results:
    print(movie)

**9. $exists: 이 연산자는 특정 필드가 문서에 존재하는지 여부에 따라 문서를 선택합니다.**

In [79]:
for movie in movies.find({'writers': {'$exists': False}}).limit(3):
    print(movie)

{'_id': ObjectId('573a1390f29313caabcd4135'), 'plot': 'Three men hammer on an anvil and pass a bottle of beer around.', 'genres': ['Short'], 'runtime': 1, 'cast': ['Charles Kayser', 'John Ott'], 'num_mflix_comments': 1, 'title': 'Blacksmith Scene', 'fullplot': 'A stationary camera looks at a large anvil with a blacksmith behind it and one on either side. The smith in the middle draws a heated metal rod from the fire, places it on the anvil, and all three begin a rhythmic hammering. After several blows, the metal goes back in the fire. One smith pulls out a bottle of beer, and they each take a swig. Then, out comes the glowing metal and the hammering resumes.', 'countries': ['USA'], 'released': datetime.datetime(1893, 5, 9, 0, 0), 'directors': ['William K.L. Dickson'], 'rated': 'UNRATED', 'awards': {'wins': 1, 'nominations': 0, 'text': '1 win.'}, 'lastupdated': '2015-08-26 00:03:50.133000000', 'year': 1893, 'imdb': {'rating': 6.2, 'votes': 1189, 'id': 5}, 'type': 'movie', 'tomatoes': {'

**10. count_documents: 이 메소드는 쿼리에 일치하는 문서의 수를 반환합니다.**
- find() 대신에 count_documents() 메서드로 count 값을 얻을 수 있음

> find().count() 방식도 문서의 수를 세는 방법으로 사용할 수 있지만, 이 방식은 MongoDB 4.0 이후로 공식적으로 deprecated (사용이 권장되지 않는) 되었습니다.

In [ ]:
## genres 필드에 "Action" 또는 "Adventure"가 포함된 영화의 갯수 세기
count = movies.count_documents({"genres":{"$in":["Action", "Adventure"]}})
print(count)

3805
